In [1]:
import os
import numpy as np
import cv2
from tqdm import tqdm
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt


In [2]:
IMG_SIZE = 224

train_dir = 'D:\\Fake money detection\\Egyptian new currency\\dataset\\train'
val_dir = 'D:\\Fake money detection\\Egyptian new currency\\dataset\\valid'
test_dir = 'D:\\Fake money detection\\Egyptian new currency\\dataset\\test'

class_names = sorted([dir_name for dir_name in os.listdir(train_dir)])
print("Classes:", class_names)


Classes: ['1', '10', '10 (new)', '100', '20', '20 (new)', '200', '5', '50']


In [3]:
def enhance_image(img):
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl,a,b))
    img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


In [4]:
def read_images_from_folder(folder_path):
    images = []
    labels = []

    for class_folder in tqdm(os.listdir(folder_path)):
        class_path = os.path.join(folder_path, class_folder)
        if os.path.isdir(class_path):
            for image_file in os.listdir(class_path):
                image_path = os.path.join(class_path, image_file)
                try:
                    img = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
                    img_array = enhance_image(np.array(img, dtype=np.uint8))
                    images.append(img_array)
                    labels.append(class_folder)
                except Exception as e:
                    print(f"Error reading image: {image_path} - {e}")
    return np.array(images), np.array(labels)


In [5]:
train_images, train_labels = read_images_from_folder(train_dir)
val_images, val_labels = read_images_from_folder(val_dir)
test_images, test_labels = read_images_from_folder(test_dir)

print(f"Train: {train_images.shape}, Val: {val_images.shape}, Test: {test_images.shape}")


100%|██████████| 9/9 [01:59<00:00, 13.31s/it]

Train: (2637, 224, 224, 3), Val: (760, 224, 224, 3), Test: (290, 224, 224, 3)


In [6]:
le = LabelEncoder()
train_labels_enc = to_categorical(le.fit_transform(train_labels))
val_labels_enc = to_categorical(le.transform(val_labels))
test_labels_enc = to_categorical(le.transform(test_labels))


In [7]:
train_datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    horizontal_flip=True,
    brightness_range=(0.8, 1.2),
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator()

train_generator = train_datagen.flow(train_images, train_labels_enc, batch_size=32)
val_generator = val_datagen.flow(val_images, val_labels_enc, batch_size=32)


In [8]:
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
predictions = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,061,100 (15.49 MB)

 Trainable params: 11,529 (45.04 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [9]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop]
)


Epoch 1/20


c:\Python\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


83/83 ━━━━━━━━━━━━━━━━━━━━ 82s 864ms/step - accuracy: 0.1203 - loss: 2.2847 - val_accuracy: 0.2013 - val_loss: 2.0889
Epoch 2/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 70s 814ms/step - accuracy: 0.1974 - loss: 2.1029 - val_accuracy: 0.3066 - val_loss: 1.9350
Epoch 3/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 68s 800ms/step - accuracy: 0.2690 - loss: 1.9780 - val_accuracy: 0.4039 - val_loss: 1.8103
Epoch 4/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 73s 861ms/step - accuracy: 0.3502 - loss: 1.8411 - val_accuracy: 0.4658 - val_loss: 1.7040
Epoch 5/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 71s 834ms/step - accuracy: 0.4019 - loss: 1.7520 - val_accuracy: 0.5303 - val_loss: 1.6098
Epoch 6/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 70s 822ms/step - accuracy: 0.4365 - loss: 1.6860 - val_accuracy: 0.5671 - val_loss: 1.5285
Epoch 7/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 71s 830ms/step - accuracy: 0.4771 - loss: 1.6028 - val_accuracy: 0.6000 - val_loss: 1.4571
Epoch 8/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 70s 816ms/step - accuracy: 0.5392 - loss: 1.5038 - val_accuracy: 0.614

In [10]:
test_datagen = ImageDataGenerator()
test_generator = test_datagen.flow(test_images, test_labels_enc, batch_size=32)

loss, acc = model.evaluate(test_generator)
print(f"Test Accuracy: {acc * 100:.2f}%")


10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 518ms/step - accuracy: 0.7780 - loss: 0.9513
Test Accuracy: 77.93%
